In [38]:
import pandas as pd
import numpy as np
import pickle
from google.colab import files

In [45]:
print("Upload these files:")
print("1. xgboost.pkl (the actual trained model)")
print("2. feature_engineered_data (4).csv (or the correct name of your feature engineered data)")

uploaded = files.upload()

Upload these files:
1. xgboost.pkl (the actual trained model)
2. feature_engineered_data (4).csv (or the correct name of your feature engineered data)


Saving feature_engineered_data.csv to feature_engineered_data (5).csv
Saving xgboost.pkl to xgboost (1).pkl


In [46]:
import pickle

with open("xgboost.pkl", "rb") as f:
    model = pickle.load(f)

print("Best model loaded successfully!")
print(f"Type of loaded model: {type(model)}")

Best model loaded successfully!
Type of loaded model: <class 'xgboost.sklearn.XGBRegressor'>


In [47]:
if not hasattr(model, "predict"):
    raise TypeError(
        "ERROR: The loaded file is NOT a trained model. "
        "Please ensure you uploaded the actual XGBoost model (.pkl)."
    )

# -------------------------------
# Load Dataset
# -------------------------------

df = pd.read_csv("feature_engineered_data (4).csv")

# -------------------------------
# Change these if your column names are different
# -------------------------------

DATE_COLUMN = "date"
TARGET_COLUMN = "electricity_units_kwh"

df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])

future_date = df[DATE_COLUMN].max() + pd.Timedelta(days=1)

future = pd.DataFrame({
    "date": [future_date]
})

In [50]:
future["year"] = future["date"].dt.year
future["month"] = future["date"].dt.month
future["day"] = future["date"].dt.day
future["day_of_week"] = future["date"].dt.dayofweek
future["is_weekend"] = (future["day_of_week"] >= 5).astype(int)

future["lag_1"] = df[TARGET_COLUMN].iloc[-1]
future["lag_7"] = df[TARGET_COLUMN].iloc[-7]
future["lag_30"] = df[TARGET_COLUMN].iloc[-30]

future["rolling_mean_7"] = df[TARGET_COLUMN].tail(7).mean()
future["rolling_mean_30"] = df[TARGET_COLUMN].tail(30).mean()

In [51]:
features = [
    "year",
    "month",
    "day",
    "day_of_week",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_30",
    "rolling_mean_7",
    "rolling_mean_30"
]

X_future = future[features]

In [60]:
future["city"] = df["city"].iloc[-1]
future["temperature_c"] = df["temperature_c"].iloc[-1]
future["humidity_percent"] = df["humidity_percent"].iloc[-1]
future["household_size"] = df["household_size"].iloc[-1]
future["income_level"] = df["income_level"].iloc[-1]
future["power_outage_hours"] = df["power_outage_hours"].iloc[-1]

future["year"] = future["date"].dt.year
future["month"] = future["date"].dt.month
future["day"] = future["date"].dt.day
future["day_of_week"] = future["date"].dt.dayofweek
future["is_weekend"] = (future["day_of_week"] >= 5).astype(int)

future["lag_1"] = df["electricity_units_kwh"].iloc[-1]
future["lag_7"] = df["electricity_units_kwh"].iloc[-7]
future["lag_30"] = df["electricity_units_kwh"].iloc[-30]

future["rolling_mean_7"] = df["electricity_units_kwh"].tail(7).mean()
future["rolling_mean_30"] = df["electricity_units_kwh"].tail(30).mean()

In [62]:
features = model.feature_names_in_

X_future = future[features]

prediction = model.predict(X_future)

In [63]:
print(future)

        date  year  month  day  day_of_week  is_weekend  city  temperature_c  \
0 2026-02-09  2026      2    9            0           0     3           32.4   

   humidity_percent  household_size  income_level  power_outage_hours  lag_1  \
0                68               6             1                0.31  20.67   

   lag_7  lag_30  rolling_mean_7  rolling_mean_30  
0  22.08    21.4       20.601429        22.177667  


In [65]:
future.to_csv("future_prediction.csv", index=False)
print("\nCSV Saved Successfully!")

# -------------------------------
# Download CSV
# ------------------------------
files.download("future_prediction.csv")



CSV Saved Successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>